# Reset Derived Data to the Three-Year Bronze Baseline

This maintenance notebook removes obsolete **price-derived test data** and rebuilds Silver, Gold, and metric datasets from the completed three-year `prices_eod` Bronze ingestion.

Bronze files are authoritative and protected. Form 4 Silver, security dimensions, Cosmos control-plane state, and connector watermarks are preserved; Form 4 Gold is rebuilt from current Silver.

## 1. Configure the Three-Year Bronze Baseline

Set the authoritative source, expected date range, reset scope, and safety switches. The notebook defaults to dry-run mode. Destructive execution requires `dry_run = True` and `execute_reset = False`.

In [ ]:
from datetime import date, datetime, timedelta, timezone
import json
import uuid

from delta.tables import DeltaTable
from pyspark.sql import Window
from pyspark.sql import functions as F

# Environment and authoritative Bronze baseline
environment = "dev"
workspace_id = "00000000-0000-4000-8000-000000000001"
lakehouse_id = "00000000-0000-4000-8000-000000000002"
lakehouse_name = "auspex_bronze"
source_id = "prices_eod"
bronze_source_path = f"Files/bronze/{source_id}"
baseline_from_date = "2023-07-02"
baseline_to_date = "2026-07-14"
expected_schema_version = 1

# Safety switches: both conditions are required for deletion and rebuild.
dry_run = True
execute_reset = False

# Rebuild controls
silver_notebook = "nb_02_prices_to_silver"
gold_notebook = "nb_03_silver_to_gold"
anchor_notebook = "nb_09_fundamental_anchor"
metrics_notebook = "nb_04_metrics"
notebook_timeout_seconds = 7200
anchor_batch_days = 31
metrics_max_runs = 130
audit_table = "ops_reset_audit"

if date.fromisoformat(baseline_from_date) > date.fromisoformat(baseline_to_date):
    raise ValueError("baseline_from_date must be on or before baseline_to_date")
if execute_reset and dry_run:
    raise ValueError("Set dry_run=False before enabling execute_reset")

reset_enabled = execute_reset and not dry_run
execution_id = str(uuid.uuid4())
started_at = datetime.now(timezone.utc)
protected_paths = [bronze_source_path]

# Derived test-era objects that are fully rebuildable from current Silver/Bronze.
price_tables = [
    "silver_prices",
    "fact_market_daily",
    "fact_insider_txn",
    "fact_fundamental_anchor",
    "security_daily_features",
]
shared_rebuild_tables = ["dim_date", "dim_entity"]
serving_projections = [
    "v_market_momentum",
    "v_market_risk",
    "v_risk_adjusted",
    "v_smart_money",
    "v_opportunity_legs",
    "v_opportunity_score",
    "v_security_daily_features",
    "v_fundamental_anchor",
]
quarantine_tables = [
    "silver_security_quarantine",
    "silver_dq_quarantine",
    "silver_parse_errors",
]
checkpoint_roots = [
    f"Files/checkpoints/{source_id}",
    f"Files/tmp/{source_id}",
    f"Files/staging/{source_id}",
]

run_state = {
    "status": "CONFIGURED",
    "critical_errors": [],
    "inventory": {},
    "bronze_validation": {},
    "deletion_plan": {},
    "removal_log": {},
    "rebuild_log": {},
    "post_rebuild_validation": {},
}

print(
    f"Execution {execution_id} | environment={environment} | source={source_id} | "
    f"baseline={baseline_from_date}..{baseline_to_date} | dry_run={dry_run} | "
    f"execute_reset={execute_reset}"
)
print(f"Protected Bronze path: {bronze_source_path}")

## 2. Inventory Existing Lakehouse Data

Capture Bronze file inventory, derived-table schemas and row counts, quarantine rows, and price-specific checkpoint or staging paths before any destructive action.

In [ ]:
def list_files_recursive(root_path):
    try:
        entries = mssparkutils.fs.ls(root_path)
    except Exception:
        return []
    files = []
    pending = list(entries)
    while pending:
        entry = pending.pop()
        if entry.isDir:
            try:
                pending.extend(mssparkutils.fs.ls(entry.path))
            except Exception:
                pass
        else:
            files.append({"path": entry.path, "size": int(entry.size)})
    return files

bronze_files = (
    spark.read.format("binaryFile")
    .option("recursiveFileLookup", "true")
    .option("pathGlobFilter", "*.ndjson")
    .load(bronze_source_path)
    .select("path", "length", "modificationTime")
    .cache()
)
bronze_file_summary = bronze_files.agg(
    F.count("*").alias("files"),
    F.sum("length").alias("bytes"),
    F.min("modificationTime").alias("first_modified_at"),
    F.max("modificationTime").alias("last_modified_at"),
).first()

inventory_rows = []
for object_name in price_tables + shared_rebuild_tables + serving_projections:
    exists = spark.catalog.tableExists(object_name)
    row_count = spark.table(object_name).count() if exists else None
    schema_json = spark.table(object_name).schema.json() if exists else None
    inventory_rows.append((object_name, exists, row_count, schema_json))

quarantine_inventory = {}
for table_name in quarantine_tables:
    if spark.catalog.tableExists(table_name):
        quarantine_inventory[table_name] = (
            spark.table(table_name)
            .filter(F.col("source_id") == source_id)
            .count()
        )

checkpoint_inventory = {
    root_path: list_files_recursive(root_path)
    for root_path in checkpoint_roots
}

run_state["inventory"] = {
    "bronze_files": int(bronze_file_summary.files or 0),
    "bronze_bytes": int(bronze_file_summary.bytes or 0),
    "bronze_first_modified_at": (
        bronze_file_summary.first_modified_at.isoformat()
        if bronze_file_summary.first_modified_at else None
    ),
    "bronze_last_modified_at": (
        bronze_file_summary.last_modified_at.isoformat()
        if bronze_file_summary.last_modified_at else None
    ),
    "tables": {
        row[0]: {"exists": row[1], "rows": row[2], "schema_json": row[3]}
        for row in inventory_rows
    },
    "price_quarantine_rows": quarantine_inventory,
    "checkpoint_files": {
        root: {"files": len(files), "bytes": sum(item["size"] for item in files)}
        for root, files in checkpoint_inventory.items()
    },
}

inventory_df = spark.createDataFrame(
    inventory_rows,
    "object_name STRING, exists BOOLEAN, row_count LONG, schema_json STRING",
)
display(inventory_df.orderBy("object_name"))
display(bronze_files.orderBy("path"))
print(json.dumps(run_state["inventory"], indent=2, sort_keys=True))

## 3. Validate Bronze Ingestion Completeness

Validate the authoritative Bronze files before deletion: JSON integrity, envelope fields, schema version, event-date coverage, business keys, duplicate payloads, provider revisions, and suspicious gaps. Critical failures prevent reset execution.

In [ ]:
bronze_paths = [row.path for row in bronze_files.select("path").collect()]
if not bronze_paths:
    run_state["critical_errors"].append("No prices_eod Bronze NDJSON files were found")

raw_lines = (
    spark.read.text(bronze_paths)
    .select(F.col("value").alias("raw_json"), F.input_file_name().alias("input_file"))
)
raw_prices = (
    raw_lines
    .withColumn("root_json", F.get_json_object("raw_json", "$"))
    .withColumn("envelope_source", F.get_json_object("raw_json", "$.source_id"))
    .withColumn("schema_version", F.get_json_object("raw_json", "$.schema_version").cast("int"))
    .withColumn("batch_id", F.get_json_object("raw_json", "$.batch_id"))
    .withColumn("ingest_ts", F.to_timestamp(F.get_json_object("raw_json", "$.ingest_ts")))
    .withColumn("symbol", F.upper(F.get_json_object("raw_json", "$.record.symbol")))
    .withColumn("price_date", F.to_date(F.get_json_object("raw_json", "$.record.date")))
    .withColumn("open", F.get_json_object("raw_json", "$.record.open").cast("decimal(18,6)"))
    .withColumn("high", F.get_json_object("raw_json", "$.record.high").cast("decimal(18,6)"))
    .withColumn("low", F.get_json_object("raw_json", "$.record.low").cast("decimal(18,6)"))
    .withColumn("close", F.get_json_object("raw_json", "$.record.close").cast("decimal(18,6)"))
    .withColumn("adj_close", F.get_json_object("raw_json", "$.record.adj_close").cast("decimal(18,6)"))
    .withColumn("volume", F.get_json_object("raw_json", "$.record.volume").cast("long"))
    .withColumn(
        "price_revision_hash",
        F.sha2(
            F.to_json(F.struct(
                "symbol",
                F.date_format("price_date", "yyyy-MM-dd").alias("date"),
                "open", "high", "low", "close", "adj_close", "volume",
            )),
            256,
        ),
    )
    .cache()
)

bronze_revisions = (
    raw_prices
    .filter(
        F.col("symbol").isNotNull()
        & F.col("price_date").isNotNull()
        & F.col("price_revision_hash").isNotNull()
    )
    .dropDuplicates(["symbol", "price_date", "price_revision_hash"])
    .cache()
)

raw_summary = raw_prices.agg(
    F.count("*").alias("raw_rows"),
    F.sum(F.col("root_json").isNull().cast("long")).alias("corrupt_rows"),
    F.sum((F.col("envelope_source") != source_id).cast("long")).alias("source_mismatches"),
    F.sum((F.col("schema_version") != expected_schema_version).cast("long")).alias("schema_mismatches"),
    F.sum(
        (
            F.col("batch_id").isNull()
            | F.col("ingest_ts").isNull()
            | F.col("symbol").isNull()
            | F.col("price_date").isNull()
        ).cast("long")
    ).alias("null_required_keys"),
    F.min("price_date").alias("min_event_date"),
    F.max("price_date").alias("max_event_date"),
).first()
revision_rows = bronze_revisions.count()
natural_keys = bronze_revisions.select("symbol", "price_date").distinct().count()
revised_keys = (
    bronze_revisions
    .groupBy("symbol", "price_date")
    .agg(F.countDistinct("price_revision_hash").alias("revisions"))
    .filter(F.col("revisions") > 1)
    .count()
)

price_dates = bronze_revisions.select("price_date").distinct()
date_window = Window.orderBy("price_date")
largest_event_gap_days = (
    price_dates
    .withColumn("previous_date", F.lag("price_date").over(date_window))
    .withColumn("gap_days", F.datediff("price_date", "previous_date") - 1)
    .agg(F.coalesce(F.max("gap_days"), F.lit(0)).alias("largest_gap"))
    .first()
    .largest_gap
)

schema_versions = [
    row.schema_version
    for row in raw_prices.select("schema_version").distinct().orderBy("schema_version").collect()
]
validation = {
    "raw_rows": int(raw_summary.raw_rows or 0),
    "payload_revisions": int(revision_rows),
    "natural_keys": int(natural_keys),
    "exact_repeat_rows": int((raw_summary.raw_rows or 0) - revision_rows),
    "revised_natural_keys": int(revised_keys),
    "corrupt_rows": int(raw_summary.corrupt_rows or 0),
    "source_mismatches": int(raw_summary.source_mismatches or 0),
    "schema_mismatches": int(raw_summary.schema_mismatches or 0),
    "null_required_keys": int(raw_summary.null_required_keys or 0),
    "schema_versions": schema_versions,
    "min_event_date": raw_summary.min_event_date.isoformat() if raw_summary.min_event_date else None,
    "max_event_date": raw_summary.max_event_date.isoformat() if raw_summary.max_event_date else None,
    "largest_event_gap_days": int(largest_event_gap_days or 0),
}
run_state["bronze_validation"] = validation

baseline_start = date.fromisoformat(baseline_from_date)
baseline_end = date.fromisoformat(baseline_to_date)
if validation["raw_rows"] == 0:
    run_state["critical_errors"].append("Bronze contains no rows")
for metric_name in ["corrupt_rows", "source_mismatches", "schema_mismatches", "null_required_keys"]:
    if validation[metric_name]:
        run_state["critical_errors"].append(f"Bronze {metric_name}={validation[metric_name]}")
if not raw_summary.min_event_date or (raw_summary.min_event_date - baseline_start).days > 7:
    run_state["critical_errors"].append("Bronze does not reach the expected baseline start")
if not raw_summary.max_event_date or (baseline_end - raw_summary.max_event_date).days > 7:
    run_state["critical_errors"].append("Bronze does not reach the expected baseline end")
if validation["largest_event_gap_days"] > 10:
    run_state["critical_errors"].append(
        f"Bronze contains an unexpected {validation['largest_event_gap_days']}-day event gap"
    )

validation_df = spark.createDataFrame(
    [(key, json.dumps(value) if isinstance(value, (list, dict)) else str(value)) for key, value in validation.items()],
    "check_name STRING, check_value STRING",
)
display(validation_df)
print(f"Critical Bronze validation errors: {run_state['critical_errors']}")

## 4. Preview Obsolete Data Removal

Build and display the exact deletion plan. In dry-run mode this is the final destructive boundary: no table, file, checkpoint, or Bronze object is modified.

In [ ]:
deletion_rows = []
for object_name in serving_projections + price_tables + shared_rebuild_tables:
    object_inventory = run_state["inventory"]["tables"].get(object_name, {})
    deletion_rows.append((
        "lakehouse_object",
        object_name,
        bool(object_inventory.get("exists", False)),
        object_inventory.get("rows"),
        None,
        object_name not in protected_paths,
    ))

for table_name, row_count in run_state["inventory"]["price_quarantine_rows"].items():
    deletion_rows.append((
        "quarantine_rows",
        f"{table_name}:source_id={source_id}",
        row_count > 0,
        row_count,
        None,
        True,
    ))

for root_path, files in checkpoint_inventory.items():
    deletion_rows.append((
        "checkpoint_or_temp_path",
        root_path,
        bool(files),
        None,
        sum(item["size"] for item in files),
        root_path not in protected_paths and not root_path.startswith(bronze_source_path),
    ))

plan_df = spark.createDataFrame(
    deletion_rows,
    "object_type STRING, object_name STRING, exists BOOLEAN, estimated_rows LONG, estimated_bytes LONG, approved BOOLEAN",
)
run_state["deletion_plan"] = {
    "objects": [
        {
            "object_type": row.object_type,
            "object_name": row.object_name,
            "exists": row.exists,
            "estimated_rows": row.estimated_rows,
            "estimated_bytes": row.estimated_bytes,
            "approved": row.approved,
        }
        for row in plan_df.collect()
    ],
    "protected_paths": protected_paths,
}

display(plan_df.orderBy("object_type", "object_name"))
if not reset_enabled:
    print("DRY RUN: deletion plan generated; no derived data or files will be removed.")
if run_state["critical_errors"]:
    print("RESET BLOCKED: Bronze validation contains critical errors.")

## 5. Remove Legacy Derived Data and Checkpoints

When explicitly enabled, delete only approved rebuildable downstream objects, price-specific quarantine rows, and exact checkpoint/staging paths. Verify that Bronze file count and bytes remain unchanged and that Form 4 Silver/security objects remain present.

In [ ]:
preserved_before = {
    "silver_insider_txn": spark.catalog.tableExists("silver_insider_txn"),
    "dim_security": spark.catalog.tableExists("dim_security"),
}
removal_log = {
    "deleted_quarantine_rows": {},
    "dropped_objects": [],
    "removed_paths": [],
    "bronze_before": {
        "files": int(bronze_file_summary.files or 0),
        "bytes": int(bronze_file_summary.bytes or 0),
    },
}

if reset_enabled and not run_state["critical_errors"]:
    try:
        for table_name in quarantine_tables:
            if not spark.catalog.tableExists(table_name):
                continue
            source_rows = (
                spark.table(table_name)
                .filter(F.col("source_id") == source_id)
                .count()
            )
            if source_rows:
                DeltaTable.forName(spark, table_name).delete(f"source_id = '{source_id}'")
            removal_log["deleted_quarantine_rows"][table_name] = int(source_rows)

        for object_name in serving_projections + price_tables + shared_rebuild_tables:
            dropped = False
            for object_type in ("VIEW", "TABLE"):
                try:
                    spark.sql(f"DROP {object_type} IF EXISTS {object_name}")
                    dropped = True
                except Exception:
                    pass
            if dropped:
                removal_log["dropped_objects"].append(object_name)

        for root_path, files in checkpoint_inventory.items():
            if not files:
                continue
            if root_path in protected_paths or root_path.startswith(bronze_source_path):
                raise RuntimeError(f"Refusing to remove protected path: {root_path}")
            mssparkutils.fs.rm(root_path, True)
            removal_log["removed_paths"].append(root_path)

        still_present = [
            object_name
            for object_name in serving_projections + price_tables + shared_rebuild_tables
            if spark.catalog.tableExists(object_name)
        ]
        if still_present:
            raise RuntimeError(f"Derived objects remain after reset: {still_present}")

        preserved_after = {
            object_name: spark.catalog.tableExists(object_name)
            for object_name in preserved_before
        }
        missing_preserved = [
            object_name
            for object_name, existed in preserved_before.items()
            if existed and not preserved_after[object_name]
        ]
        if missing_preserved:
            raise RuntimeError(f"Preserved objects disappeared: {missing_preserved}")

        bronze_after = (
            spark.read.format("binaryFile")
            .option("recursiveFileLookup", "true")
            .option("pathGlobFilter", "*.ndjson")
            .load(bronze_source_path)
            .agg(F.count("*").alias("files"), F.sum("length").alias("bytes"))
            .first()
        )
        removal_log["bronze_after"] = {
            "files": int(bronze_after.files or 0),
            "bytes": int(bronze_after.bytes or 0),
        }
        if removal_log["bronze_before"] != removal_log["bronze_after"]:
            raise RuntimeError("Protected Bronze inventory changed during reset")
        run_state["status"] = "RESET_COMPLETE"
    except Exception as exc:
        run_state["status"] = "RESET_FAILED"
        run_state["critical_errors"].append(f"Reset failed: {exc}")
else:
    run_state["status"] = "PREVIEW_COMPLETE" if not run_state["critical_errors"] else "VALIDATION_FAILED"

run_state["removal_log"] = removal_log
print(json.dumps(removal_log, indent=2, sort_keys=True))

In [ ]:
# Remove only stale Form 4 Silver facts whose resolved security no longer exists.
stale_form4_silver_rows = 0
if reset_enabled and run_state["status"] == "RESET_COMPLETE":
    if spark.catalog.tableExists("silver_insider_txn") and spark.catalog.tableExists("dim_security"):
        orphan_form4_keys = (
            spark.table("silver_insider_txn").alias("txn")
            .join(
                spark.table("dim_security").select("security_sk").distinct().alias("security"),
                F.col("txn.security_sk") == F.col("security.security_sk"),
                "left_anti",
            )
            .select("accession_no", "line_no")
            .distinct()
            .cache()
        )
        stale_form4_silver_rows = orphan_form4_keys.count()
        if stale_form4_silver_rows:
            (
                DeltaTable.forName(spark, "silver_insider_txn")
                .alias("target")
                .merge(
                    orphan_form4_keys.alias("stale"),
                    "target.accession_no = stale.accession_no AND target.line_no = stale.line_no",
                )
                .whenMatchedDelete()
                .execute()
            )
        orphan_form4_keys.unpersist()
        remaining_form4_orphans = (
            spark.table("silver_insider_txn").alias("txn")
            .join(
                spark.table("dim_security").select("security_sk").distinct().alias("security"),
                F.col("txn.security_sk") == F.col("security.security_sk"),
                "left_anti",
            )
            .count()
        )
        if remaining_form4_orphans:
            raise RuntimeError(
                f"Stale Form 4 Silver cleanup failed: remaining_orphans={remaining_form4_orphans}"
            )
run_state["removal_log"]["deleted_stale_form4_silver_rows"] = stale_form4_silver_rows
print(f"Deleted stale Form 4 Silver rows: {stale_form4_silver_rows}")

## 6. Rebuild Tables from the Bronze Baseline

Orchestrate the canonical revision-safe transformation notebooks instead of duplicating business logic. Notebook 02 rebuilds Silver from Bronze, Notebook 03 rebuilds Gold, Notebook 09 rebuilds the PIT fundamental anchor, and Notebook 04 runs in bounded batches until all stale metric snapshot dates converge.

In [ ]:
def parse_notebook_result(value):
    if not value:
        return None
    try:
        return json.loads(value)
    except (TypeError, json.JSONDecodeError):
        return value


def remaining_stale_metric_dates():
    if not spark.catalog.tableExists("fact_market_daily"):
        return None
    market_updates = (
        spark.table("fact_market_daily")
        .filter(
            F.col("event_date").isNotNull()
            & F.col("knowledge_date").isNotNull()
            & F.col("revision_loaded_at").isNotNull()
        )
        .withColumn("as_of", F.greatest("event_date", "knowledge_date"))
        .groupBy("as_of")
        .agg(F.max("revision_loaded_at").alias("date_revision_loaded_at"))
    )
    freshness_window = Window.orderBy("as_of").rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow,
    )
    source_freshness = market_updates.withColumn(
        "source_revision_loaded_at",
        F.max("date_revision_loaded_at").over(freshness_window),
    )
    if not spark.catalog.tableExists("security_daily_features"):
        return source_freshness.count()
    feature_freshness = (
        spark.table("security_daily_features")
        .groupBy("as_of")
        .agg(F.min("feature_built_at").alias("feature_built_at"))
    )
    return (
        source_freshness
        .join(feature_freshness, "as_of", "left")
        .filter(
            F.col("feature_built_at").isNull()
            | (F.col("source_revision_loaded_at") > F.col("feature_built_at"))
        )
        .count()
    )


rebuild_log = {"silver": None, "gold": None, "anchor": [], "metrics_runs": []}
if reset_enabled and run_state["status"] == "RESET_COMPLETE":
    try:
        silver_result = mssparkutils.notebook.run(
            silver_notebook,
            notebook_timeout_seconds,
            {"from_date": baseline_from_date, "to_date": baseline_to_date},
        )
        rebuild_log["silver"] = parse_notebook_result(silver_result)

        gold_result = mssparkutils.notebook.run(
            gold_notebook,
            notebook_timeout_seconds,
            {},
        )
        rebuild_log["gold"] = parse_notebook_result(gold_result)

        anchor_start = date.fromisoformat(baseline_from_date)
        anchor_end = date.fromisoformat(baseline_to_date)
        while anchor_start <= anchor_end:
            anchor_window_end = min(
                anchor_start + timedelta(days=anchor_batch_days - 1),
                anchor_end,
            )
            anchor_result = mssparkutils.notebook.run(
                anchor_notebook,
                notebook_timeout_seconds,
                {
                    "from_date": anchor_start.isoformat(),
                    "to_date": anchor_window_end.isoformat(),
                    "max_anchor_dates": anchor_batch_days,
                },
            )
            rebuild_log["anchor"].append({
                "from_date": anchor_start.isoformat(),
                "to_date": anchor_window_end.isoformat(),
                "result": parse_notebook_result(anchor_result),
            })
            anchor_start = anchor_window_end + timedelta(days=1)

        remaining = remaining_stale_metric_dates()
        for run_number in range(1, metrics_max_runs + 1):
            if remaining == 0:
                break
            metrics_result = mssparkutils.notebook.run(
                metrics_notebook,
                notebook_timeout_seconds,
                {},
            )
            remaining = remaining_stale_metric_dates()
            rebuild_log["metrics_runs"].append({
                "run_number": run_number,
                "result": parse_notebook_result(metrics_result),
                "remaining_stale_snapshot_dates": remaining,
            })
        if remaining != 0:
            raise RuntimeError(
                f"Metrics did not converge after {metrics_max_runs} runs: remaining={remaining}"
            )
        run_state["status"] = "REBUILD_COMPLETE"
    except Exception as exc:
        run_state["status"] = "REBUILD_FAILED"
        run_state["critical_errors"].append(f"Rebuild failed: {exc}")
else:
    print("Rebuild skipped because this is a dry run or reset validation failed.")

run_state["rebuild_log"] = rebuild_log
print(json.dumps(rebuild_log, indent=2, sort_keys=True, default=str))

## 7. Run Post-Rebuild Data Quality Checks

Verify Bronze preservation, rebuilt row counts, revision grain, required columns, null/PIT rules, Gold/Silver convergence, metric freshness, and rejected records. Critical failures are recorded for the audit before the notebook exits with an error.

In [ ]:
post_validation = {}
if reset_enabled and run_state["status"] == "REBUILD_COMPLETE":
    try:
        bronze_post = (
            spark.read.format("binaryFile")
            .option("recursiveFileLookup", "true")
            .option("pathGlobFilter", "*.ndjson")
            .load(bronze_source_path)
            .agg(F.count("*").alias("files"), F.sum("length").alias("bytes"))
            .first()
        )
        post_validation["bronze_files"] = int(bronze_post.files or 0)
        post_validation["bronze_bytes"] = int(bronze_post.bytes or 0)
        post_validation["bronze_unchanged"] = (
            post_validation["bronze_files"] == run_state["inventory"]["bronze_files"]
            and post_validation["bronze_bytes"] == run_state["inventory"]["bronze_bytes"]
        )

        silver_df = spark.table("silver_prices").cache()
        silver_counts = silver_df.agg(
            F.count("*").alias("rows"),
            F.countDistinct("symbol", "date").alias("natural_keys"),
        ).first()
        post_validation["silver_rows"] = int(silver_counts.rows)
        post_validation["silver_natural_keys"] = int(silver_counts.natural_keys)
        post_validation["silver_revised_keys"] = (
            silver_df
            .groupBy("symbol", "date")
            .agg(F.countDistinct("price_revision_hash").alias("revisions"))
            .filter(F.col("revisions") > 1)
            .count()
        )
        post_validation["silver_duplicate_revision_keys"] = (
            silver_df
            .groupBy("security_sk", "date", "price_revision_hash")
            .count()
            .filter(F.col("count") > 1)
            .count()
        )
        post_validation["silver_missing_required"] = silver_df.filter(
            F.col("security_sk").isNull()
            | F.col("date").isNull()
            | F.col("price_revision_hash").isNull()
            | F.col("knowledge_date").isNull()
            | F.col("ingest_ts").isNull()
            | F.col("revision_loaded_at").isNull()
        ).count()
        silver_required_columns = {
            "security_sk", "symbol", "date", "price_revision_hash", "open", "high",
            "low", "close", "adj_close", "volume", "event_date", "knowledge_date",
            "batch_id", "ingest_ts", "revision_loaded_at",
        }
        post_validation["silver_missing_columns"] = sorted(
            silver_required_columns.difference(silver_df.columns)
        )

        market_df = spark.table("fact_market_daily").cache()
        post_validation["market_rows"] = market_df.count()
        post_validation["market_duplicate_revision_keys"] = (
            market_df
            .groupBy("security_sk", "date_sk", "price_revision_hash")
            .count()
            .filter(F.col("count") > 1)
            .count()
        )
        post_validation["market_missing_required"] = market_df.filter(
            F.col("security_sk").isNull()
            | F.col("date_sk").isNull()
            | F.col("price_revision_hash").isNull()
            | F.col("event_date").isNull()
            | F.col("knowledge_date").isNull()
            | F.col("ingest_ts").isNull()
            | F.col("revision_loaded_at").isNull()
        ).count()

        features_df = spark.table("security_daily_features").cache()
        post_validation["feature_rows"] = features_df.count()
        post_validation["feature_duplicate_keys"] = (
            features_df
            .groupBy("security_sk", "date_sk")
            .count()
            .filter(F.col("count") > 1)
            .count()
        )
        post_validation["feature_invalid_pit"] = features_df.filter(
            F.col("as_of").isNull()
            | F.col("max_knowledge_date").isNull()
            | (F.col("max_knowledge_date") > F.col("as_of"))
            | F.col("feature_built_at").isNull()
        ).count()
        post_validation["remaining_stale_snapshot_dates"] = remaining_stale_metric_dates()

        rejected_rows = {}
        for table_name in quarantine_tables:
            if spark.catalog.tableExists(table_name):
                rejected_rows[table_name] = (
                    spark.table(table_name)
                    .filter(F.col("source_id") == source_id)
                    .count()
                )
        post_validation["rejected_rows"] = rejected_rows

        if not post_validation["bronze_unchanged"]:
            run_state["critical_errors"].append("Bronze inventory changed during rebuild")
        if post_validation["silver_rows"] <= 0:
            run_state["critical_errors"].append("Rebuilt silver_prices is empty")
        if post_validation["silver_revised_keys"] > validation["revised_natural_keys"]:
            run_state["critical_errors"].append("Silver contains revisions absent from current Bronze")
        if post_validation["silver_duplicate_revision_keys"]:
            run_state["critical_errors"].append("Silver contains duplicate revision keys")
        if post_validation["silver_missing_required"] or post_validation["silver_missing_columns"]:
            run_state["critical_errors"].append("Silver required fields or columns are missing")
        if post_validation["market_rows"] != post_validation["silver_rows"]:
            run_state["critical_errors"].append("Gold market row count does not match Silver")
        if post_validation["market_duplicate_revision_keys"] or post_validation["market_missing_required"]:
            run_state["critical_errors"].append("Gold market revision validation failed")
        if post_validation["feature_rows"] <= 0:
            run_state["critical_errors"].append("Rebuilt security_daily_features is empty")
        if post_validation["feature_duplicate_keys"] or post_validation["feature_invalid_pit"]:
            run_state["critical_errors"].append("Feature grain or PIT validation failed")
        if post_validation["remaining_stale_snapshot_dates"] != 0:
            run_state["critical_errors"].append("Metric snapshots did not converge")

        silver_df.unpersist()
        market_df.unpersist()
        features_df.unpersist()
    except Exception as exc:
        run_state["critical_errors"].append(f"Post-rebuild validation failed: {exc}")
else:
    post_validation["skipped"] = True

run_state["post_rebuild_validation"] = post_validation
post_validation_df = spark.createDataFrame(
    [(key, json.dumps(value, sort_keys=True) if isinstance(value, (list, dict)) else str(value)) for key, value in post_validation.items()],
    "check_name STRING, check_value STRING",
)
display(post_validation_df)
print(f"Post-rebuild critical errors: {run_state['critical_errors']}")

## 8. Persist the Reset Audit Report

Append an immutable execution receipt containing parameters, pre-reset inventory, Bronze validation, deletion plan, removals, rebuild results, post-rebuild checks, timestamps, final status, and any error. Audit persistence occurs before a critical failure is raised.

In [ ]:
completed_at = datetime.now(timezone.utc)
if run_state["critical_errors"]:
    final_status = "FAILED"
elif reset_enabled:
    final_status = "REBUILD_SUCCEEDED"
else:
    final_status = "DRY_RUN_VALIDATED"

parameters = {
    "environment": environment,
    "workspace_id": workspace_id,
    "lakehouse_id": lakehouse_id,
    "lakehouse_name": lakehouse_name,
    "source_id": source_id,
    "bronze_source_path": bronze_source_path,
    "baseline_from_date": baseline_from_date,
    "baseline_to_date": baseline_to_date,
    "expected_schema_version": expected_schema_version,
    "dry_run": dry_run,
    "execute_reset": execute_reset,
    "silver_notebook": silver_notebook,
    "gold_notebook": gold_notebook,
    "anchor_notebook": anchor_notebook,
    "anchor_batch_days": anchor_batch_days,
    "metrics_notebook": metrics_notebook,
    "metrics_max_runs": metrics_max_runs,
}

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {audit_table} (
        execution_id                 STRING    NOT NULL,
        environment                  STRING    NOT NULL,
        source_id                    STRING    NOT NULL,
        baseline_from_date           DATE      NOT NULL,
        baseline_to_date             DATE      NOT NULL,
        dry_run                      BOOLEAN   NOT NULL,
        execute_reset                BOOLEAN   NOT NULL,
        started_at                   TIMESTAMP NOT NULL,
        completed_at                 TIMESTAMP NOT NULL,
        final_status                 STRING    NOT NULL,
        parameters_json              STRING    NOT NULL,
        inventory_json               STRING    NOT NULL,
        bronze_validation_json       STRING    NOT NULL,
        deletion_plan_json           STRING    NOT NULL,
        removal_log_json             STRING    NOT NULL,
        rebuild_log_json             STRING    NOT NULL,
        post_rebuild_validation_json STRING    NOT NULL,
        errors_json                  STRING    NOT NULL
    )
    USING DELTA
""")

audit_row = [(
    execution_id,
    environment,
    source_id,
    date.fromisoformat(baseline_from_date),
    date.fromisoformat(baseline_to_date),
    dry_run,
    execute_reset,
    started_at.replace(tzinfo=None),
    completed_at.replace(tzinfo=None),
    final_status,
    json.dumps(parameters, sort_keys=True),
    json.dumps(run_state["inventory"], sort_keys=True, default=str),
    json.dumps(run_state["bronze_validation"], sort_keys=True, default=str),
    json.dumps(run_state["deletion_plan"], sort_keys=True, default=str),
    json.dumps(run_state["removal_log"], sort_keys=True, default=str),
    json.dumps(run_state["rebuild_log"], sort_keys=True, default=str),
    json.dumps(run_state["post_rebuild_validation"], sort_keys=True, default=str),
    json.dumps(run_state["critical_errors"], sort_keys=True),
)]
audit_schema = """
    execution_id STRING, environment STRING, source_id STRING,
    baseline_from_date DATE, baseline_to_date DATE,
    dry_run BOOLEAN, execute_reset BOOLEAN,
    started_at TIMESTAMP, completed_at TIMESTAMP, final_status STRING,
    parameters_json STRING, inventory_json STRING, bronze_validation_json STRING,
    deletion_plan_json STRING, removal_log_json STRING, rebuild_log_json STRING,
    post_rebuild_validation_json STRING, errors_json STRING
"""
spark.createDataFrame(audit_row, audit_schema).write.format("delta").mode("append").saveAsTable(audit_table)

final_summary = {
    "execution_id": execution_id,
    "final_status": final_status,
    "dry_run": dry_run,
    "execute_reset": execute_reset,
    "bronze_modified": False,
    "bronze_validation": run_state["bronze_validation"],
    "removal_log": run_state["removal_log"],
    "rebuild_log": run_state["rebuild_log"],
    "post_rebuild_validation": run_state["post_rebuild_validation"],
    "critical_errors": run_state["critical_errors"],
    "audit_table": audit_table,
}
final_summary_json = json.dumps(final_summary, sort_keys=True, default=str)
print(f"THREE-YEAR BASELINE RESET SUMMARY: {final_summary_json}", flush=True)

bronze_revisions.unpersist()
raw_prices.unpersist()
bronze_files.unpersist()

if run_state["critical_errors"]:
    raise RuntimeError(f"THREE-YEAR BASELINE RESET FAILED: {final_summary_json}")

mssparkutils.notebook.exit(final_summary_json)